# ETF_V5：固定正交 ETF 池下的 ML vs 手动趋势模型对比

目标：在 V4 生成的固定当前正交 ETF 池上，复跑 ML vs 手动趋势模型、horizon、训练窗口和风险层对比。

本实验只比较四类核心变量：

- `score_family`：机器学习 LGB 模型 vs 手动 R²/趋势模型
- `horizon_days`：预测/评估 `ret3d`、`ret5d`、`ret10d`
- `train_window`：`2021-2023`、`2019-2024`、`2021-2025`
- `risk_layer`：无风险层 vs ETF 趋势广度差时切现金

说明：

- 组合选择默认只做原始 top1/top3，不做行业/主题去重；本 notebook 固定读取 V4 正交 ETF 池。
- 离线 proxy 使用 feature date 收盘到 horizon 后收盘的收益，适合筛方向，不完全等同 JoinQuant 09:35 成交回测。
- 风险层使用股票型/行业 ETF 池自身趋势广度，低于阈值时收益按现金 `0` 处理；实际回测可映射到货币 ETF 或空仓。
- 训练样本只使用 `next_date_h <= train_end` 的已知标签，避免训练截止日穿越。

In [ ]:
# =========================
# 0. Config
# =========================
from jqdata import *
import os
import gc
import math
import pickle
import datetime
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

OUT_DIR = "etf_ml_v5_fixed_orthogonal_pool_outputs"
USE_ORTHOGONAL_POOL = True
ORTHOGONAL_POOL_PATH = "etf_v4_fixed_orthogonal_pool_outputs/etf_orthogonal_pool_v1.csv"
PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v5_weekly_panel.csv")
SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v5_score_panel.csv")
WEEKLY_CSV = os.path.join(OUT_DIR, "etf_ml_v5_weekly_portfolio_proxy.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v5_summary.csv")
LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v5_latest_targets.csv")
RISK_REPORT_CSV = os.path.join(OUT_DIR, "etf_ml_v5_risk_layer_report.csv")
MODEL_MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v5_model_manifest.csv")

REBUILD_DATA = True
EXPORT_MODELS = False
START_DATE = "2019-01-01"
END_DATE = "2026-06-30"
LOOKBACK_DAYS = 60
HORIZON_DAYS_LIST = [3, 5, 10]
MAX_HORIZON_DAYS = max(HORIZON_DAYS_LIST)
MIN_LISTING_DAYS = 180
MIN_AVG_MONEY_20 = 20000000.0
ETF_CHUNK_SIZE = 120
BENCHMARK = "000985.XSHG"
RANDOM_SEED = 42

# 这版先不做行业/主题硬限制；top3 就是模型/手动分数原始 top3。
SELECT_MODES = ["top1", "top3"]
RISK_MODES = ["none", "breadth_cash"]
RISK_BREADTH_COL = "pool_breadth_25"
RISK_BREADTH_THRESHOLD = 0.35
CASH_RETURN = 0.0

TREND_WINDOWS = [10, 20, 25, 60]
TREND_WEIGHT_END = 2.0
PRICE_HISTORY_COUNT = max(LOOKBACK_DAYS + 1, max(TREND_WINDOWS) + 1)
POOL_CONTEXT_WINDOW = 25

EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单",
    "短融", "中票", "AAA", "信用", "0-3", "1-3", "政策性金融债",
]
ETF_NAME_CACHE = {}

TRAIN_WINDOWS = [
    ("train20210101_20231231", "2021-01-01", "2023-12-31"),
    ("train20190101_20241231", "2019-01-01", "2024-12-31"),
    ("train20210101_20251231", "2021-01-01", "2025-12-31"),
]

BASE_PRICE_FEATURE_COLS = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "vol_5", "vol_20", "vol_60",
    "close_to_ma20", "close_to_ma60", "ma5_to_ma20", "ma20_to_ma60",
    "drawdown_20", "drawdown_60",
    "amp_20", "amp_60",
    "money_mean_20", "money_ratio_5_20", "money_ratio_20_60",
    "volume_ratio_5_20", "volume_ratio_20_60",
    "max_ret_20", "min_ret_20",
]
TREND_FEATURE_COLS = []
for w in TREND_WINDOWS:
    TREND_FEATURE_COLS.extend([
        "trend_ann_%s" % w,
        "trend_r2_%s" % w,
        "trend_score_%s" % w,
        "trend_vol_%s" % w,
        "trend_score_vol_adj_%s" % w,
        "trend_simple_ann_%s" % w,
    ])
CONTEXT_FEATURE_COLS = [
    "pool_breadth_%s" % POOL_CONTEXT_WINDOW,
    "pool_median_vol_%s" % POOL_CONTEXT_WINDOW,
]
RAW_FEATURE_COLS = BASE_PRICE_FEATURE_COLS + TREND_FEATURE_COLS
RANK_FEATURE_COLS = ["rank_" + c for c in RAW_FEATURE_COLS]
FEATURE_COLS = RAW_FEATURE_COLS + RANK_FEATURE_COLS + CONTEXT_FEATURE_COLS

META_COLS = ["code", "name", "feature_date", "rebalance_date"]
for h in HORIZON_DAYS_LIST:
    META_COLS.extend(["future_ret_%sd" % h, "next_date_%sd" % h])

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 31,
    "max_depth": 5,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.90,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "min_gain_to_split": 0.0,
    "verbose": -1,
    "seed": RANDOM_SEED,
}
NUM_BOOST_ROUND = 180

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

print("out dir:", OUT_DIR)
print("use orthogonal pool:", USE_ORTHOGONAL_POOL, ORTHOGONAL_POOL_PATH)
print("features:", len(FEATURE_COLS))
print("horizons:", HORIZON_DAYS_LIST)
print("train windows:", [x[0] for x in TRAIN_WINDOWS])
print("risk threshold:", RISK_BREADTH_COL, "<", RISK_BREADTH_THRESHOLD)

In [ ]:
# =========================
# 0b. Fixed current orthogonal ETF pool
# =========================
def load_orthogonal_pool(path):
    if not USE_ORTHOGONAL_POOL:
        return pd.DataFrame(), set()
    if not os.path.exists(path):
        raise IOError("orthogonal pool file not found: " + path + "\nRun ETF_V4_fixed_current_orthogonal_pool_builder实验.ipynb first. V5 requires the fixed orthogonal pool.")
    pool = pd.read_csv(path)
    if "code" not in pool.columns:
        raise ValueError("orthogonal pool csv missing code column: " + path)
    pool["code"] = pool["code"].astype(str)
    codes = set(pool["code"].dropna().astype(str).tolist())
    if len(codes) == 0:
        raise ValueError("orthogonal pool is empty: " + path)
    return pool, codes


def filter_to_orthogonal_pool(df):
    if (not USE_ORTHOGONAL_POOL) or df is None or df.empty:
        return df
    if "code" not in df.columns:
        return df
    before = df.shape[0]
    out = df[df["code"].astype(str).isin(ORTHOGONAL_POOL_CODES)].copy()
    print("orthogonal pool filter rows:", before, "->", out.shape[0])
    return out


orthogonal_pool_df, ORTHOGONAL_POOL_CODES = load_orthogonal_pool(ORTHOGONAL_POOL_PATH)
print("orthogonal pool size:", len(ORTHOGONAL_POOL_CODES))
if USE_ORTHOGONAL_POOL:
    display(orthogonal_pool_df.head(30))

In [ ]:
# =========================
# 1. Data helpers: JoinQuant weekly ETF panel
# =========================
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def get_weekly_feature_dates(start_date, end_date):
    try:
        days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    except NameError:
        raise RuntimeError("数据重建需要在 JoinQuant 研究环境运行；如果已有缓存，请设置 REBUILD_DATA=False。")
    if len(days) == 0:
        return []
    s = pd.Series(days)
    out = []
    for _, gdf in s.groupby(s.dt.strftime("%Y-%W")):
        out.append(pd.Timestamp(gdf.max()).normalize())
    return out


def should_exclude_name(name):
    text = str(name)
    for kw in EXCLUDE_NAME_KEYWORDS:
        if kw and kw in text:
            return True
    return False


def get_etf_universe_on_date(date):
    try:
        sec_df = get_all_securities(["etf"], date=date)
    except NameError:
        raise RuntimeError("get_all_securities 不可用：请在 JoinQuant 研究环境运行数据重建。")
    except Exception as err:
        print("get_all_securities failed", date, err)
        return []
    if sec_df is None or sec_df.empty:
        return []
    out = []
    name_map = {}
    for code, row in sec_df.iterrows():
        try:
            start_date = row.get("start_date", None)
            if pd.isnull(start_date):
                start_date = get_security_info(code).start_date
            start_date = pd.Timestamp(start_date).date()
            feature_dt = pd.Timestamp(date).date()
            if feature_dt - start_date < datetime.timedelta(days=MIN_LISTING_DAYS):
                continue
            name = row.get("display_name", "")
            if should_exclude_name(name):
                continue
            out.append(code)
            name_map[code] = str(name)
        except Exception:
            continue
    if USE_ORTHOGONAL_POOL:
        out = [code for code in out if str(code) in ORTHOGONAL_POOL_CODES]
        name_map = dict((code, name_map.get(code, "")) for code in out)
    ETF_NAME_CACHE[str(pd.Timestamp(date).date())] = name_map
    return out


def safe_ratio(a, b):
    if pd.isnull(a) or pd.isnull(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)


def safe_ret(close, days):
    if len(close) <= days:
        return np.nan
    base = close.iloc[-days - 1]
    if pd.isnull(base) or base <= 0:
        return np.nan
    return close.iloc[-1] / base - 1.0


def calc_trend_metrics(close, days):
    empty = {"ann": np.nan, "r2": np.nan, "score": np.nan, "vol": np.nan, "score_vol_adj": np.nan, "simple_ann": np.nan}
    if len(close) <= days:
        return empty
    recent = pd.Series(close.iloc[-(days + 1):].astype(float).values)
    if recent.isnull().any() or (recent <= 0).any():
        return empty
    y = np.log(recent.values)
    x = np.arange(len(y))
    weights = np.linspace(1.0, TREND_WEIGHT_END, len(y))
    try:
        slope, intercept = np.polyfit(x, y, 1, w=weights)
    except Exception:
        return empty
    ann = math.exp(slope * 250.0) - 1.0
    fit = slope * x + intercept
    ss_res = np.sum(weights * (y - fit) ** 2)
    ss_tot = np.sum(weights * (y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else 0.0
    r2 = max(0.0, min(1.0, float(r2)))
    daily_returns = recent.pct_change().dropna()
    vol = float(daily_returns.std() * math.sqrt(250.0)) if len(daily_returns) > 1 else np.nan
    period_ret = recent.iloc[-1] / recent.iloc[0] - 1.0
    simple_ann = (1.0 + period_ret) ** (250.0 / float(days)) - 1.0 if 1.0 + period_ret > 0 else np.nan
    score = ann * r2
    score_vol_adj = score * 0.20 / max(vol, 0.05) if not pd.isnull(vol) else np.nan
    return {"ann": ann, "r2": r2, "score": score, "vol": vol, "score_vol_adj": score_vol_adj, "simple_ann": simple_ann}


def calc_one_etf_features(code, price_df):
    df = price_df.sort_values("time").copy()
    if len(df) < max(40, int(LOOKBACK_DAYS * 0.8)):
        return None
    for col in ["open", "high", "low", "close", "volume", "money"]:
        if col not in df.columns:
            return None
    close = pd.Series(df["close"].astype(float).values)
    high = pd.Series(df["high"].astype(float).values)
    low = pd.Series(df["low"].astype(float).values)
    volume = pd.Series(df["volume"].astype(float).values)
    money = pd.Series(df["money"].astype(float).values)
    if close.isnull().any() or close.iloc[-1] <= 0:
        return None

    ret = close.pct_change()
    rec = {"code": code}
    rec["ret_1"] = safe_ret(close, 1)
    rec["ret_5"] = safe_ret(close, 5)
    rec["ret_10"] = safe_ret(close, 10)
    rec["ret_20"] = safe_ret(close, 20)
    rec["ret_60"] = safe_ret(close, 60)
    rec["vol_5"] = ret.tail(5).std()
    rec["vol_20"] = ret.tail(20).std()
    rec["vol_60"] = ret.tail(60).std()
    rec["close_to_ma20"] = safe_ratio(close.iloc[-1], close.tail(20).mean()) - 1.0
    rec["close_to_ma60"] = safe_ratio(close.iloc[-1], close.tail(60).mean()) - 1.0
    rec["ma5_to_ma20"] = safe_ratio(close.tail(5).mean(), close.tail(20).mean()) - 1.0
    rec["ma20_to_ma60"] = safe_ratio(close.tail(20).mean(), close.tail(60).mean()) - 1.0
    rec["drawdown_20"] = safe_ratio(close.iloc[-1], close.tail(20).max()) - 1.0
    rec["drawdown_60"] = safe_ratio(close.iloc[-1], close.tail(60).max()) - 1.0
    rec["amp_20"] = (high.tail(20) / low.tail(20) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["amp_60"] = (high.tail(60) / low.tail(60) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["money_mean_20"] = money.tail(20).mean()
    rec["money_ratio_5_20"] = safe_ratio(money.tail(5).mean(), money.tail(20).mean()) - 1.0
    rec["money_ratio_20_60"] = safe_ratio(money.tail(20).mean(), money.tail(60).mean()) - 1.0
    rec["volume_ratio_5_20"] = safe_ratio(volume.tail(5).mean(), volume.tail(20).mean()) - 1.0
    rec["volume_ratio_20_60"] = safe_ratio(volume.tail(20).mean(), volume.tail(60).mean()) - 1.0
    rec["max_ret_20"] = ret.tail(20).max()
    rec["min_ret_20"] = ret.tail(20).min()
    for w in TREND_WINDOWS:
        tm = calc_trend_metrics(close, w)
        rec["trend_ann_%s" % w] = tm["ann"]
        rec["trend_r2_%s" % w] = tm["r2"]
        rec["trend_score_%s" % w] = tm["score"]
        rec["trend_vol_%s" % w] = tm["vol"]
        rec["trend_score_vol_adj_%s" % w] = tm["score_vol_adj"]
        rec["trend_simple_ann_%s" % w] = tm["simple_ann"]
    return rec


def fetch_feature_rows(feature_date, etfs):
    rows = []
    fields = ["open", "high", "low", "close", "volume", "money"]
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            price_df = get_price(
                etf_chunk,
                end_date=feature_date,
                frequency="daily",
                fields=fields,
                count=PRICE_HISTORY_COUNT,
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("price feature chunk failed", feature_date, err)
            continue
        if price_df is None or price_df.empty:
            continue
        for code, one in price_df.groupby("code"):
            rec = calc_one_etf_features(code, one)
            if rec is not None:
                rows.append(rec)
    if len(rows) == 0:
        return pd.DataFrame()
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)


def fetch_future_returns_multi(feature_date, etfs):
    ret_cols = ["future_ret_%sd" % h for h in HORIZON_DAYS_LIST]
    next_cols = ["next_date_%sd" % h for h in HORIZON_DAYS_LIST]
    try:
        td = pd.to_datetime(get_trade_days(start_date=feature_date, count=MAX_HORIZON_DAYS + 1))
    except Exception as err:
        print("future trade days failed", feature_date, err)
        return pd.DataFrame(columns=["code"] + ret_cols + next_cols)
    if len(td) < MAX_HORIZON_DAYS + 1:
        return pd.DataFrame(columns=["code"] + ret_cols + next_cols)
    next_date_map = {}
    for h in HORIZON_DAYS_LIST:
        next_date_map[h] = pd.Timestamp(td[h]).normalize()
    max_next_date = next_date_map[MAX_HORIZON_DAYS]

    rows = []
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            px = get_price(
                etf_chunk,
                start_date=feature_date,
                end_date=max_next_date,
                frequency="daily",
                fields=["close"],
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("future price chunk failed", feature_date, err)
            continue
        if px is None or px.empty:
            continue
        for code, one in px.groupby("code"):
            one = one.sort_values("time").copy()
            if len(one) < 2:
                continue
            one["time_norm"] = pd.to_datetime(one["time"]).dt.normalize()
            close_by_date = dict(zip(one["time_norm"], one["close"].astype(float)))
            start_dt = pd.Timestamp(feature_date).normalize()
            if start_dt not in close_by_date or close_by_date[start_dt] <= 0:
                continue
            start_close = float(close_by_date[start_dt])
            rec = {"code": code}
            ok = False
            for h in HORIZON_DAYS_LIST:
                nd = next_date_map[h]
                rec["next_date_%sd" % h] = nd
                if nd in close_by_date:
                    rec["future_ret_%sd" % h] = float(close_by_date[nd]) / start_close - 1.0
                    ok = True
                else:
                    rec["future_ret_%sd" % h] = np.nan
            if ok:
                rows.append(rec)
    return pd.DataFrame(rows)


def add_rank_features(df):
    out = df.copy()
    for col in RAW_FEATURE_COLS:
        if col in out.columns:
            out["rank_" + col] = out[col].rank(pct=True)
    return out


def add_pool_context_features(df):
    out = df.copy()
    ann_col = "trend_ann_%s" % POOL_CONTEXT_WINDOW
    vol_col = "trend_vol_%s" % POOL_CONTEXT_WINDOW
    if ann_col in out.columns and len(out) > 0:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = float((out[ann_col] > 0).mean())
    else:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = np.nan
    if vol_col in out.columns and len(out) > 0:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = float(out[vol_col].median())
    else:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = np.nan
    return out


def build_one_feature_date(feature_date):
    etfs = get_etf_universe_on_date(feature_date)
    if len(etfs) == 0:
        return pd.DataFrame()
    feature_df = fetch_feature_rows(feature_date, etfs)
    if feature_df.empty:
        return pd.DataFrame()
    if "money_mean_20" in feature_df.columns:
        feature_df = feature_df[feature_df["money_mean_20"] >= MIN_AVG_MONEY_20].copy()
    if feature_df.empty:
        return pd.DataFrame()
    feature_df = add_pool_context_features(feature_df)
    label_df = fetch_future_returns_multi(feature_date, list(feature_df["code"]))
    if label_df.empty:
        return pd.DataFrame()
    out = feature_df.merge(label_df, on="code", how="inner")
    if out.empty:
        return out
    name_map = ETF_NAME_CACHE.get(str(pd.Timestamp(feature_date).date()), {})
    out["name"] = out["code"].map(name_map).fillna("")
    out = add_rank_features(out)
    out["feature_date"] = pd.Timestamp(feature_date).normalize()
    out["rebalance_date"] = out["feature_date"]
    return out

In [ ]:
# =========================
# 2. Build or load weekly ETF panel
# =========================
def drop_incomplete_label_dates(panel):
    out = panel.copy()
    bad_dates = set()
    for h in HORIZON_DAYS_LIST:
        col = "future_ret_%sd" % h
        if col not in out.columns:
            continue
        q = out.groupby("feature_date")[col].agg(["count", lambda s: (s == 0).mean()])
        q.columns = ["sample_count", "zero_return_rate"]
        for dt in q[q["zero_return_rate"] >= 0.98].index:
            bad_dates.add(dt)
    if bad_dates:
        print("drop incomplete label dates:", sorted([str(pd.Timestamp(x).date()) for x in bad_dates]))
        out = out[~out["feature_date"].isin(bad_dates)].copy()
    return out


def find_excluded_name_rows(df, name_col):
    if df is None or df.empty or name_col not in df.columns:
        return pd.DataFrame()
    mask = pd.Series(False, index=df.index)
    names = df[name_col].astype(str)
    for kw in EXCLUDE_NAME_KEYWORDS:
        mask = mask | names.str.contains(kw, na=False)
    return df.loc[mask].copy()


def assert_no_excluded_names(df, name_col, label):
    bad = find_excluded_name_rows(df, name_col)
    if bad.empty:
        print(label + " excluded-name check: OK")
        return
    cols = []
    for c in ["feature_date", "code", "name", "targets", "target_names", "model_name", "select_mode", "risk_mode"]:
        if c in bad.columns and c not in cols:
            cols.append(c)
    print(label + " excluded-name check failed, rows=", len(bad))
    if cols:
        print(bad[cols].head(20).to_string(index=False))
    raise ValueError(label + " contains excluded ETF names. Rebuild data or clean old cache before trusting outputs.")


def build_weekly_panel():
    feature_dates = get_weekly_feature_dates(START_DATE, END_DATE)
    print("feature weeks:", len(feature_dates), "from", feature_dates[0] if feature_dates else None, "to", feature_dates[-1] if feature_dates else None)
    parts = []
    for dt in tqdm(feature_dates, desc="build etf weekly panel"):
        one = build_one_feature_date(dt)
        if one is not None and not one.empty:
            parts.append(one)
        if len(parts) % 20 == 0:
            gc.collect()
    if len(parts) == 0:
        raise RuntimeError("No ETF weekly samples were built. Check ETF universe/data access.")
    panel = pd.concat(parts, ignore_index=True)
    for c in ["feature_date", "rebalance_date"]:
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    for h in HORIZON_DAYS_LIST:
        c = "next_date_%sd" % h
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    return drop_incomplete_label_dates(panel)


if REBUILD_DATA or (not os.path.exists(PANEL_CSV)):
    panel_df = build_weekly_panel()
    panel_df.to_csv(PANEL_CSV, index=False)
else:
    panel_df = pd.read_csv(PANEL_CSV)
    date_cols = ["feature_date", "rebalance_date"] + ["next_date_%sd" % h for h in HORIZON_DAYS_LIST]
    for c in date_cols:
        if c in panel_df.columns:
            panel_df[c] = pd.to_datetime(panel_df[c])

panel_df = filter_to_orthogonal_pool(panel_df)
assert_no_excluded_names(panel_df, "name", "panel_df")
print("panel shape:", panel_df.shape)
print(panel_df[["feature_date"] + ["next_date_%sd" % h for h in HORIZON_DAYS_LIST]].agg(["min", "max"]))
print("sample per week:")
print(panel_df.groupby("feature_date")["code"].count().describe())
print("missing labels:")
print(panel_df[["future_ret_%sd" % h for h in HORIZON_DAYS_LIST]].isnull().mean())
display(panel_df.head())

In [ ]:
# =========================
# 3. Scoring models: ML LGB and manual R2/trend score
# =========================
def clean_feature_frame(df):
    cols = []
    for c in META_COLS + FEATURE_COLS:
        if c in df.columns and c not in cols:
            cols.append(c)
    return df.loc[:, cols].replace([np.inf, -np.inf], np.nan).copy()


def train_lgb_or_fallback(train_df, target_col):
    train_df = train_df[~train_df[target_col].isnull()].copy()
    X_raw = train_df.reindex(columns=FEATURE_COLS).replace([np.inf, -np.inf], np.nan)
    fill_values = X_raw.median().to_dict()
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0)
    y = train_df[target_col].astype(float).values
    try:
        import lightgbm as lgb
        dtrain = lgb.Dataset(X[FEATURE_COLS], label=y, feature_name=list(FEATURE_COLS), free_raw_data=True)
        model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
        backend = "lightgbm"
        del dtrain
    except Exception as err:
        print("LightGBM unavailable or failed, fallback to sklearn RandomForestRegressor:", err)
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor(
            n_estimators=240,
            max_depth=6,
            min_samples_leaf=25,
            random_state=RANDOM_SEED,
            n_jobs=1,
        )
        model.fit(X[FEATURE_COLS], y)
        backend = "sklearn_random_forest"
    del X_raw, X, y
    gc.collect()
    return model, fill_values, backend


def predict_model(model, fill_values, df):
    X_raw = df.reindex(columns=FEATURE_COLS).replace([np.inf, -np.inf], np.nan)
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0)
    pred = np.asarray(model.predict(X[FEATURE_COLS])).reshape(-1).astype(float)
    del X_raw, X
    gc.collect()
    return pred


def calc_manual_trend_score(df):
    out = pd.Series(0.0, index=df.index)
    weights = [
        ("rank_trend_score_vol_adj_25", 0.40),
        ("rank_trend_score_20", 0.20),
        ("rank_trend_r2_25", 0.15),
        ("rank_ret_20", 0.10),
        ("rank_drawdown_20", 0.10),
        ("rank_money_ratio_5_20", 0.05),
        ("rank_vol_20", -0.10),
    ]
    total_abs = 0.0
    for col, w in weights:
        if col in df.columns:
            out = out + df[col].fillna(df[col].median()).fillna(0.5).astype(float) * w
            total_abs += abs(w)
    if total_abs == 0:
        return pd.Series(0.0, index=df.index)
    return out


def make_bundle(model, fill_values, backend, horizon_days, train_tag, train_start, train_end):
    return {
        "objective": "etf_ml_v3_weekly_lgb_ret_horizon",
        "model": model,
        "model_backend": backend,
        "horizon_days": int(horizon_days),
        "target_col": "future_ret_%sd" % horizon_days,
        "feature_cols": list(FEATURE_COLS),
        "raw_feature_cols": list(RAW_FEATURE_COLS),
        "rank_feature_cols": list(RANK_FEATURE_COLS),
        "context_feature_cols": list(CONTEXT_FEATURE_COLS),
        "fill_values": dict(fill_values),
        "lookback_days": LOOKBACK_DAYS,
        "trend_windows": list(TREND_WINDOWS),
        "price_history_count": PRICE_HISTORY_COUNT,
        "min_avg_money_20": MIN_AVG_MONEY_20,
        "exclude_name_keywords": list(EXCLUDE_NAME_KEYWORDS),
        "benchmark": BENCHMARK,
        "train_tag": train_tag,
        "train_start": train_start,
        "train_end": train_end,
        "research_version": "etf_ml_v3_model_vs_manual_risk_layer",
        "created_note": "No ETF industry/theme hard cap. Model compares ret3/5/10 horizons against manual trend score and breadth risk layer.",
    }

In [ ]:
# =========================
# 4. Train ML scores and build manual scores
# =========================
def append_csv(df, path):
    if df is None or df.empty:
        return
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False)


for p in [SCORE_CSV, MODEL_MANIFEST_CSV]:
    if os.path.exists(p):
        os.remove(p)

panel_df = clean_feature_frame(panel_df)
manifest_rows = []
score_keep_parts = []

for train_tag, train_start, train_end in tqdm(TRAIN_WINDOWS, desc="train windows"):
    train_start_ts = pd.Timestamp(train_start)
    train_end_ts = pd.Timestamp(train_end)
    eval_mask_base = panel_df["feature_date"] > train_end_ts
    if not bool(eval_mask_base.any()):
        print("skip eval empty", train_tag)
        continue

    eval_base = panel_df.loc[eval_mask_base].copy()
    eval_base["manual_trend_score"] = calc_manual_trend_score(eval_base)

    for h in tqdm(HORIZON_DAYS_LIST, desc=train_tag, leave=False):
        target_col = "future_ret_%sd" % h
        next_col = "next_date_%sd" % h
        train_mask = (
            (panel_df["feature_date"] >= train_start_ts) &
            (panel_df[next_col] <= train_end_ts) &
            (~panel_df[target_col].isnull())
        )
        eval_mask = eval_mask_base & (~panel_df[target_col].isnull())
        train_df = panel_df.loc[train_mask].copy()
        score_df_h = panel_df.loc[eval_mask].copy()
        if train_df.empty or score_df_h.empty:
            print("skip", train_tag, h, "train", train_df.shape, "eval", score_df_h.shape)
            continue

        print("training ML", train_tag, "ret%sd" % h, "samples", train_df.shape[0], "weeks", train_df["feature_date"].nunique())
        model, fill_values, backend = train_lgb_or_fallback(train_df, target_col)
        ml_score = predict_model(model, fill_values, score_df_h)
        manual_score = calc_manual_trend_score(score_df_h)

        base_cols = ["code", "name", "feature_date", "rebalance_date", target_col, next_col, RISK_BREADTH_COL]
        base_cols = [c for c in base_cols if c in score_df_h.columns]
        ml_out = score_df_h.loc[:, base_cols].copy()
        ml_out["score"] = ml_score
        ml_out["score_family"] = "ml_lgb"
        ml_out["horizon_days"] = h
        ml_out["train_tag"] = train_tag
        ml_out["target_col"] = target_col
        ml_out["next_date"] = ml_out[next_col]
        ml_out["future_ret"] = ml_out[target_col]
        ml_out["model_name"] = "ml_lgb_ret%sd_%s" % (h, train_tag)
        append_csv(ml_out, SCORE_CSV)
        score_keep_parts.append(ml_out)

        manual_out = score_df_h.loc[:, base_cols].copy()
        manual_out["score"] = manual_score
        manual_out["score_family"] = "manual_trend_r2"
        manual_out["horizon_days"] = h
        manual_out["train_tag"] = train_tag
        manual_out["target_col"] = target_col
        manual_out["next_date"] = manual_out[next_col]
        manual_out["future_ret"] = manual_out[target_col]
        manual_out["model_name"] = "manual_trend_r2_ret%sd_%s" % (h, train_tag)
        append_csv(manual_out, SCORE_CSV)
        score_keep_parts.append(manual_out)

        model_file = ""
        if EXPORT_MODELS:
            model_file = "model_etf_ml_v3_lgb_ret%sd_%s.pkl" % (h, train_tag)
            with open(os.path.join(OUT_DIR, model_file), "wb") as f:
                pickle.dump(make_bundle(model, fill_values, backend, h, train_tag, train_start, train_end), f, protocol=2)
        manifest_rows.append({
            "model_name": "ml_lgb_ret%sd_%s" % (h, train_tag),
            "score_family": "ml_lgb",
            "horizon_days": h,
            "train_tag": train_tag,
            "train_start": train_start,
            "train_end": train_end,
            "backend": backend,
            "model_file": model_file,
            "train_samples": train_df.shape[0],
            "train_weeks": train_df["feature_date"].nunique(),
            "eval_samples": score_df_h.shape[0],
            "eval_weeks": score_df_h["feature_date"].nunique(),
        })
        manifest_rows.append({
            "model_name": "manual_trend_r2_ret%sd_%s" % (h, train_tag),
            "score_family": "manual_trend_r2",
            "horizon_days": h,
            "train_tag": train_tag,
            "train_start": train_start,
            "train_end": train_end,
            "backend": "manual_formula",
            "model_file": "",
            "train_samples": train_df.shape[0],
            "train_weeks": train_df["feature_date"].nunique(),
            "eval_samples": score_df_h.shape[0],
            "eval_weeks": score_df_h["feature_date"].nunique(),
        })
        del model, train_df, score_df_h, ml_out, manual_out
        gc.collect()

score_df = pd.concat(score_keep_parts, ignore_index=True) if score_keep_parts else pd.DataFrame()
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MODEL_MANIFEST_CSV, index=False)
print("score shape:", score_df.shape)
display(manifest_df)

In [ ]:
# =========================
# 5. Portfolio proxy: raw top1/top3 and breadth risk layer
# =========================
def summarize_returns(ret_series):
    r = pd.Series(ret_series).dropna().astype(float)
    if len(r) == 0:
        return {"periods": 0}
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    mean_ret = float(r.mean())
    std_ret = float(r.std())
    sharpe = np.nan if std_ret <= 0 or pd.isnull(std_ret) else mean_ret / std_ret
    return {
        "periods": int(len(r)),
        "cum_ret": float(nav.iloc[-1] - 1.0),
        "mean_period_ret": mean_ret,
        "win_rate": float((r > 0).mean()),
        "period_sharpe": float(sharpe) if not pd.isnull(sharpe) else np.nan,
        "max_drawdown": float(dd.min()),
    }


def select_raw_top(sorted_df, select_mode):
    n = 1 if select_mode == "top1" else 3
    return sorted_df.head(min(n, len(sorted_df))).copy()


def apply_risk_layer(feature_df, risk_mode):
    if risk_mode == "none":
        return False
    if risk_mode == "breadth_cash":
        if RISK_BREADTH_COL not in feature_df.columns:
            return False
        breadth = pd.to_numeric(feature_df[RISK_BREADTH_COL], errors="coerce").dropna()
        if len(breadth) == 0:
            return False
        return float(breadth.iloc[0]) < RISK_BREADTH_THRESHOLD
    return False


def build_weekly_portfolio(score_data):
    rows = []
    rng = np.random.RandomState(RANDOM_SEED)
    group_cols = ["model_name", "score_family", "horizon_days", "train_tag", "feature_date"]
    for keys, gdf in tqdm(score_data.groupby(group_cols), desc="portfolio proxy"):
        model_name, score_family, h, train_tag, feature_date = keys
        gdf = gdf.sort_values("score", ascending=False).copy()
        if gdf.empty:
            continue
        median_ret = float(gdf["future_ret"].median())
        for select_mode in SELECT_MODES:
            top = select_raw_top(gdf, select_mode)
            raw_ret = float(top["future_ret"].mean()) if not top.empty else np.nan
            random_n = len(top)
            random_ret = float(gdf.sample(random_n, random_state=int(rng.randint(0, 1000000)))["future_ret"].mean()) if random_n > 0 else np.nan
            for risk_mode in RISK_MODES:
                risk_off = apply_risk_layer(gdf, risk_mode)
                realized_ret = CASH_RETURN if risk_off else raw_ret
                rows.append({
                    "model_name": model_name,
                    "score_family": score_family,
                    "horizon_days": int(h),
                    "train_tag": train_tag,
                    "feature_date": feature_date,
                    "next_date": top["next_date"].iloc[0] if not top.empty else pd.NaT,
                    "select_mode": select_mode,
                    "risk_mode": risk_mode,
                    "risk_off": bool(risk_off),
                    "pool_breadth": float(gdf[RISK_BREADTH_COL].dropna().iloc[0]) if RISK_BREADTH_COL in gdf.columns and len(gdf[RISK_BREADTH_COL].dropna()) else np.nan,
                    "target_count": int(len(top)),
                    "targets": ",".join(top["code"].astype(str).tolist()),
                    "target_names": ",".join(top.get("name", pd.Series([""] * len(top))).astype(str).tolist()),
                    "raw_portfolio_ret": raw_ret,
                    "portfolio_ret": realized_ret,
                    "universe_median_ret": median_ret,
                    "excess_vs_median": realized_ret - median_ret if not pd.isnull(realized_ret) else np.nan,
                    "random_ret": random_ret,
                    "excess_vs_random": realized_ret - random_ret if not pd.isnull(realized_ret) and not pd.isnull(random_ret) else np.nan,
                })
    return pd.DataFrame(rows)


weekly_df = build_weekly_portfolio(score_df) if not score_df.empty else pd.DataFrame()
assert_no_excluded_names(weekly_df, "target_names", "weekly_df")
weekly_df.to_csv(WEEKLY_CSV, index=False)
print("weekly proxy shape:", weekly_df.shape)
display(weekly_df.head())

In [ ]:
# =========================
# 6. Summary and diagnostics
# =========================
def build_summary(weekly):
    rows = []
    if weekly.empty:
        return pd.DataFrame()
    group_cols = ["score_family", "horizon_days", "train_tag", "select_mode", "risk_mode", "model_name"]
    for keys, gdf in weekly.groupby(group_cols):
        score_family, h, train_tag, select_mode, risk_mode, model_name = keys
        ret_stats = summarize_returns(gdf["portfolio_ret"])
        med_stats = summarize_returns(gdf["excess_vs_median"])
        rnd_stats = summarize_returns(gdf["excess_vs_random"])
        row = {
            "score_family": score_family,
            "horizon_days": int(h),
            "train_tag": train_tag,
            "select_mode": select_mode,
            "risk_mode": risk_mode,
            "model_name": model_name,
            "risk_off_rate": float(gdf["risk_off"].mean()),
            "avg_pool_breadth": float(gdf["pool_breadth"].mean()),
        }
        row.update({"ret_" + k: v for k, v in ret_stats.items()})
        row.update({"excess_median_" + k: v for k, v in med_stats.items()})
        row.update({"excess_random_" + k: v for k, v in rnd_stats.items()})
        rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["ret_cum_ret", "ret_max_drawdown"], ascending=[False, False])
    return out


def build_risk_report(weekly):
    rows = []
    if weekly.empty:
        return pd.DataFrame()
    key_cols = ["score_family", "horizon_days", "train_tag", "select_mode", "model_name"]
    base = weekly[weekly["risk_mode"] == "none"].copy()
    risk = weekly[weekly["risk_mode"] == "breadth_cash"].copy()
    for keys, g0 in base.groupby(key_cols):
        mask = pd.Series(True, index=risk.index)
        for col, val in zip(key_cols, keys):
            mask = mask & (risk[col] == val)
        g1 = risk[mask].copy()
        if g1.empty:
            continue
        s0 = summarize_returns(g0.sort_values("feature_date")["portfolio_ret"])
        s1 = summarize_returns(g1.sort_values("feature_date")["portfolio_ret"])
        rows.append({
            "score_family": keys[0],
            "horizon_days": keys[1],
            "train_tag": keys[2],
            "select_mode": keys[3],
            "model_name": keys[4],
            "risk_off_rate": float(g1["risk_off"].mean()),
            "cum_ret_none": s0.get("cum_ret", np.nan),
            "cum_ret_breadth_cash": s1.get("cum_ret", np.nan),
            "cum_ret_delta": s1.get("cum_ret", np.nan) - s0.get("cum_ret", np.nan),
            "max_dd_none": s0.get("max_drawdown", np.nan),
            "max_dd_breadth_cash": s1.get("max_drawdown", np.nan),
            "max_dd_improvement": s1.get("max_drawdown", np.nan) - s0.get("max_drawdown", np.nan),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["cum_ret_delta", "max_dd_improvement"], ascending=[False, False])
    return out


summary_df = build_summary(weekly_df)
risk_report_df = build_risk_report(weekly_df)
summary_df.to_csv(SUMMARY_CSV, index=False)
risk_report_df.to_csv(RISK_REPORT_CSV, index=False)

print("Top summary by raw return:")
display(summary_df.head(30))
print("Risk layer effect:")
display(risk_report_df.head(30))

In [ ]:
# =========================
# 7. Latest targets for inspection
# =========================
latest_rows = []
if not score_df.empty:
    for (model_name, select_mode), gdf in score_df.groupby(["model_name", "score_family"]):
        latest_date = gdf["feature_date"].max()
        latest_sorted = gdf[gdf["feature_date"] == latest_date].sort_values("score", ascending=False).copy()
        for mode in SELECT_MODES:
            top = select_raw_top(latest_sorted, mode)
            for rank_idx, (_, row) in enumerate(top.iterrows(), 1):
                latest_rows.append({
                    "model_name": row["model_name"],
                    "score_family": row["score_family"],
                    "horizon_days": row["horizon_days"],
                    "train_tag": row["train_tag"],
                    "feature_date": latest_date,
                    "select_mode": mode,
                    "rank": rank_idx,
                    "code": row["code"],
                    "name": row.get("name", ""),
                    "score": row["score"],
                    "future_ret": row.get("future_ret", np.nan),
                    "pool_breadth": row.get(RISK_BREADTH_COL, np.nan),
                })
latest_targets_df = pd.DataFrame(latest_rows)
latest_targets_df.to_csv(LATEST_TARGETS_CSV, index=False)
display(latest_targets_df.head(60))

print("outputs saved:")
for p in [PANEL_CSV, SCORE_CSV, WEEKLY_CSV, SUMMARY_CSV, RISK_REPORT_CSV, LATEST_TARGETS_CSV, MODEL_MANIFEST_CSV]:
    print(" -", p)

## Self Review

- 不再使用 ETF 行业/主题硬去重；`top3` 是原始分数前三名。
- ML 训练样本用 `next_date_h <= train_end`，OOS 评估用 `feature_date > train_end`，避免训练截止日标签穿越。
- 手动趋势模型不使用未来收益，只使用 feature date 及以前构造的趋势/R²/动量/波动 rank。
- 风险层只使用同周 ETF 池内趋势广度，低于阈值时按现金收益处理；这是组合风险层，不是模型训练特征泄漏。
- 离线 proxy 是 close-to-close，用来筛方向；进入实盘/回测前，仍需用 JoinQuant 回测文件验证 09:35 成交、佣金、滑点、持仓路径。
- 长循环使用 `tqdm`，如果聚宽环境缺少 `tqdm` 会 fallback 为普通循环。